# 多状态
- State Schema: 全局状态
- Input State: 输入状态, 规范上，输入状态应当是全局状态的子集
- Output State: 输出状态, 规范上，输入状态应当是全局状态的子集
- Private State: 私有状态，规范上，私有状态的字段不应和其他状态重复，否则语义不清晰

## State Schema
- 通过`StateGraph(state_schema=xxx)`设置

In [1]:
from operator import add

from typing import Annotated, TypedDict

from langgraph.graph import StateGraph


class StateSchema(TypedDict):
    username: str
    nickname: str
    logs: Annotated[list[str], add]

## Input State
- 通过`StateGraph(input_state=xxx)`设置

In [2]:
class InputState(TypedDict):
    username: str
    logs: Annotated[list[str], add]

## Output State
- 通过`StateGraph(output_state=xxx)`设置

In [3]:
class OutputState(TypedDict):
    username: str
    logs: Annotated[list[str], add]

## Private State
- 通过节点函数输入参数指令，为节点之间传递的临时状态

In [4]:
class PrivateState(TypedDict):
    temp_val: str

In [11]:
from langgraph.constants import START, END

state_graph = StateGraph(
    state_schema=StateSchema,
    input_schema=InputState,
    output_schema=OutputState
)

def node1(input: InputState) -> OutputState:
    username = input['username']
    return {
        "username": username,
        "temp_val": "deepseek",
        "logs": [f'node_1:{username}']
    }

def node2(private_input: PrivateState) -> OutputState:
    temp_val = private_input['temp_val']
    return {
        "temp_val": temp_val,
        "logs": [f'node_2:{temp_val}']
    }

def node3(input: InputState) -> OutputState:
    username = input['username']
    return {
        "username": username,
        "logs": [f'node_3:{username}']
    }

state_graph.add_node("node1", node1)
state_graph.add_node("node2", node2)
state_graph.add_node("node3", node3)
state_graph.add_edge(START, "node1")
state_graph.add_edge("node1", "node2")
state_graph.add_edge("node2", "node3")
state_graph.add_edge("node3", END)

graph = state_graph.compile()
result = graph.invoke(
    {
        "username": "小d",
        "logs": []
    }
)
print(result)

{'username': '小d', 'logs': ['node_1:小d', 'node_2:deepseek', 'node_3:小d']}
